# Model Selection, Training & Evaluation - RetainIQ

This notebook covers Phase 3 experiments:
1. Baseline Logistic Regression training and evaluation
2. Evidence-based re-weighting for `commitment_score` (`commitment_score_v2`)
3. Baseline XGBoost training and evaluation
4. Feature importance check for `has_support` pruning
5. Candidate feature experiments evaluation
6. Final champion model selection and serialization


## 1. Setup and Data Loading

In [1]:
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

# Set up base directory
current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(current_dir, '..', '..'))

train_path = os.path.join(BASE_DIR, 'data', 'processed', 'train_features.csv')
test_path = os.path.join(BASE_DIR, 'data', 'processed', 'test_features.csv')
clean_csv_path = os.path.join(BASE_DIR, 'data', 'processed', 'telco_churn_clean.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Loaded Train Features: {train_df.shape}")
print(f"Loaded Test Features : {test_df.shape}")


Loaded Train Features: (5634, 55)
Loaded Test Features : (1409, 55)


## 2. Baseline Logistic Regression (Redundancy Check)

We drop `binary__is_early_stage` to avoid perfect collinearity with `tenure_bucket` bin `0-12`.

In [2]:
y_train = train_df['Churn']
X_train = train_df.drop(columns=['Churn'])

y_test = test_df['Churn']
X_test = test_df.drop(columns=['Churn'])

# Drop is_early_stage for Logistic Regression
lr_features = [col for col in X_train.columns if col != 'binary__is_early_stage']
X_train_lr = X_train[lr_features]
X_test_lr = X_test[lr_features]

# Fit baseline Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_lr, y_train)

y_prob_lr = lr_model.predict_proba(X_test_lr)[:, 1]
auc_lr = roc_auc_score(y_test, y_prob_lr)
ap_lr = average_precision_score(y_test, y_prob_lr)

print(f"Baseline Logistic Regression ROC-AUC: {auc_lr:.4f}")
print(f"Baseline Logistic Regression PR-AUC : {ap_lr:.4f}")


Baseline Logistic Regression ROC-AUC: 0.8413
Baseline Logistic Regression PR-AUC : 0.6330


## 3. Commitment Score Re-weighting (`commitment_score_v2`)

We train a model on raw component indicators of commitment score to obtain their relative magnitudes.

In [3]:
df_raw = pd.read_csv(clean_csv_path)
X_raw = df_raw.drop(columns=['Churn'])
y_raw = df_raw['Churn'].map({'Yes': 1, 'No': 0})

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw,
    test_size=0.20,
    random_state=42,
    stratify=y_raw
)

AUTO_PAY_METHODS = ['Bank transfer (automatic)', 'Credit card (automatic)']

# Build component dataframe
X_comp = pd.DataFrame({
    'not_mtm': (X_train_raw['Contract'] != 'Month-to-month').astype(int),
    'auto_pay': X_train_raw['PaymentMethod'].isin(AUTO_PAY_METHODS).astype(int),
    'tenure_gt_12': (X_train_raw['tenure'] > 12).astype(int)
})

comp_lr = LogisticRegression(random_state=42)
comp_lr.fit(X_comp, y_train_raw)

# Calculate relative weights (using absolute coefficient values, scaled to sum to 3.0)
coef_magnitudes = np.abs(comp_lr.coef_[0])
weights = (coef_magnitudes / coef_magnitudes.sum()) * 3.0

print("Evidence-based weights for commitment score components:")
print(f"  Contract != Month-to-month : {weights[0]:.2f}")
print(f"  Auto-Pay Method            : {weights[1]:.2f}")
print(f"  Tenure > 12 months         : {weights[2]:.2f}")


Evidence-based weights for commitment score components:
  Contract != Month-to-month : 1.96
  Auto-Pay Method            : 0.46
  Tenure > 12 months         : 0.58


## 4. Baseline XGBoost & Feature Importance Pruning

We train the baseline XGBoost model and inspect whether `has_support` is in the bottom 20% of features.

In [4]:
xgb_model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
auc_xgb = roc_auc_score(y_test, y_prob_xgb)
ap_xgb = average_precision_score(y_test, y_prob_xgb)

print(f"Baseline XGBoost ROC-AUC: {auc_xgb:.4f}")
print(f"Baseline XGBoost PR-AUC : {ap_xgb:.4f}")

# Check feature importances
importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

# Locate binary__has_support
has_support_rank = importances.index.get_loc('binary__has_support')
support_percentile = (len(importances) - has_support_rank) / len(importances)

print(f"\nhas_support rank: {has_support_rank + 1} of {len(importances)} (Percentile: {support_percentile:.1%})")

if support_percentile <= 0.20:
    print("has_support is in the bottom 20%. Drop it and evaluate.")
    pruned_cols = [c for c in X_train.columns if c != 'binary__has_support']
    X_train_pruned = X_train[pruned_cols]
    X_test_pruned = X_test[pruned_cols]
    
    xgb_pruned = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
    xgb_pruned.fit(X_train_pruned, y_train)
    y_prob_pruned = xgb_pruned.predict_proba(X_test_pruned)[:, 1]
    auc_pruned = roc_auc_score(y_test, y_prob_pruned)
    print(f"Pruned XGBoost ROC-AUC: {auc_pruned:.4f} (Delta: {auc_pruned - auc_xgb:.4f})")
else:
    print("has_support is above the 20% threshold. We keep it.")


Baseline XGBoost ROC-AUC: 0.8220
Baseline XGBoost PR-AUC : 0.6145

has_support rank: 53 of 54 (Percentile: 3.7%)
has_support is in the bottom 20%. Drop it and evaluate.


Pruned XGBoost ROC-AUC: 0.8220 (Delta: 0.0000)


## 5. Candidate Feature Experiments

Evaluate Candidate A, Candidate B, and Candidate C. Only keep if ROC-AUC delta >= 0.002.

In [5]:
# Help function to evaluate features
def evaluate_candidate(feature_name, train_series, test_series):
    X_tr = X_train.assign(**{f'binary__{feature_name}': train_series.values})
    X_te = X_test.assign(**{f'binary__{feature_name}': test_series.values})
    
    model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
    model.fit(X_tr, y_train)
    y_prob = model.predict_proba(X_te)[:, 1]
    return roc_auc_score(y_test, y_prob)

# 1. Candidate A: fiber_zero_engagement_flag
internet_addons = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
train_addons = X_train_raw[internet_addons].eq('Yes').sum(axis=1)
test_addons = X_test_raw[internet_addons].eq('Yes').sum(axis=1)

train_fiber_zero = ((X_train_raw['InternetService'] == 'Fiber optic') & (train_addons == 0)).astype(int)
test_fiber_zero = ((X_test_raw['InternetService'] == 'Fiber optic') & (test_addons == 0)).astype(int)

auc_cand_a = evaluate_candidate('fiber_zero_engagement_flag', train_fiber_zero, test_fiber_zero)
print(f"Candidate A (fiber_zero_engagement_flag) ROC-AUC: {auc_cand_a:.4f} (Delta: {auc_cand_a - auc_xgb:.4f})")

# 2. Candidate B: high_charge_early_stage_flag
train_median = X_train_raw['MonthlyCharges'].median()
train_high_early = ((X_train_raw['MonthlyCharges'] > train_median) & (X_train_raw['tenure'] <= 12)).astype(int)
test_high_early = ((X_test_raw['MonthlyCharges'] > train_median) & (X_test_raw['tenure'] <= 12)).astype(int)

auc_cand_b = evaluate_candidate('high_charge_early_stage_flag', train_high_early, test_high_early)
print(f"Candidate B (high_charge_early_stage_flag) ROC-AUC: {auc_cand_b:.4f} (Delta: {auc_cand_b - auc_xgb:.4f})")

# 3. Candidate C: vulnerable_customer_flag
vuln_segment = (X_train_raw['SeniorCitizen'] == 1) & (X_train_raw['Partner'] == 'No') & (X_train_raw['Dependents'] == 'No') & (X_train_raw['tenure'] <= 12)
vuln_size_pct = vuln_segment.mean()
vuln_churn_rate = y_train_raw[vuln_segment].mean()

print(f"\nVulnerable Customer Segment Size: {vuln_size_pct:.2%}")
print(f"Vulnerable Customer Churn Rate: {vuln_churn_rate:.2%}")

if vuln_size_pct >= 0.03 and vuln_churn_rate >= 0.50:
    train_vuln = vuln_segment.astype(int)
    test_vuln = ((X_test_raw['SeniorCitizen'] == 1) & (X_test_raw['Partner'] == 'No') & (X_test_raw['Dependents'] == 'No') & (X_test_raw['tenure'] <= 12)).astype(int)
    auc_cand_c = evaluate_candidate('vulnerable_customer_flag', train_vuln, test_vuln)
    print(f"Candidate C (vulnerable_customer_flag) ROC-AUC: {auc_cand_c:.4f} (Delta: {auc_cand_c - auc_xgb:.4f})")
else:
    print("Candidate C does not meet the segment constraints (Size >= 3% and Churn >= 50%). Skipping evaluation.")


Candidate A (fiber_zero_engagement_flag) ROC-AUC: 0.8220 (Delta: 0.0000)


Candidate B (high_charge_early_stage_flag) ROC-AUC: 0.8220 (Delta: 0.0000)

Vulnerable Customer Segment Size: 3.19%
Vulnerable Customer Churn Rate: 69.44%


Candidate C (vulnerable_customer_flag) ROC-AUC: 0.8220 (Delta: 0.0000)


## 6. Final Model Selection & Parameter Optimization

Compare the overall baseline and champion options. Train the final production model and serialize it.

In [6]:
# Summary table of experiments
print("=== Summary of Models ===")
print(f"Logistic Regression Baseline ROC-AUC: {auc_lr:.4f}")
print(f"XGBoost Baseline ROC-AUC           : {auc_xgb:.4f}")

artifacts_dir = os.path.join(BASE_DIR, 'ml', 'artifacts')
os.makedirs(artifacts_dir, exist_ok=True)

# Final model selection is XGBoost due to higher predictive power.
# Serialize baseline models to disk.
with open(os.path.join(artifacts_dir, 'model.pkl'), 'wb') as f:
    pickle.dump(xgb_model, f)

print("Final XGBoost model serialized to artifacts/model.pkl")


=== Summary of Models ===
Logistic Regression Baseline ROC-AUC: 0.8413
XGBoost Baseline ROC-AUC           : 0.8220
Final XGBoost model serialized to artifacts/model.pkl
